# Fund segmentation evaluation

This phase consumes the **saved fund registry and document classifications** from `fund_discovery_eval.ipynb`. It does not rediscover funds.

For multi-fund documents, known fund names are treated as structural anchors and converted into exclusive fund-owned page/block ranges. This prevents downstream chunks from crossing fund boundaries.


In [ ]:
from pathlib import Path
import json, sys, importlib
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from fund_rag import fund_segmentation
importlib.reload(fund_segmentation)

PARSED_DIR = ROOT / "artifacts" / "ingestion"
FUND_DISCOVERY_DIR = ROOT / "artifacts" / "fund_discovery"
SEGMENT_DIR = ROOT / "artifacts" / "fund_segmentation"
SEGMENT_DIR.mkdir(parents=True, exist_ok=True)

documents = [
    json.loads(path.read_text(encoding="utf-8"))
    for path in sorted(PARSED_DIR.glob("*.json"))
]
assert documents, "Run ingestion_eval.ipynb first."
print([d["document"] for d in documents])


## 1. Load validated discovery artifacts


In [ ]:
REGISTRY_PATH = FUND_DISCOVERY_DIR / "registry.json"
CLASSIFICATIONS_PATH = FUND_DISCOVERY_DIR / "classifications.json"

if not REGISTRY_PATH.exists() or not CLASSIFICATIONS_PATH.exists():
    raise FileNotFoundError("Run fund_discovery_eval.ipynb first.")

registry = json.loads(REGISTRY_PATH.read_text(encoding="utf-8"))
classifications = json.loads(CLASSIFICATIONS_PATH.read_text(encoding="utf-8"))

print("Loaded registry funds:", len(registry))
print("Loaded classifications:", len(classifications))


## 2. Inspect the registry and document classifications

These outputs come from the fund-discovery stage. They are inputs to segmentation.


In [ ]:
registry_df = pd.DataFrame([
    {
        "canonical_name": fund["canonical_name"],
        "aliases": " | ".join(fund.get("aliases", [])),
        "evidence_count": len(fund.get("evidence", [])),
    }
    for fund in registry
])

display(registry_df)
display(pd.DataFrame(classifications))


## 3. Confirm block-level input exists

Ingestion already created `text_blocks`. Each block preserves document-order text plus layout metadata such as font size, boldness, and bounding box.

Segmentation uses these original blocks before token chunking.


In [ ]:
prospectus = next(
    doc
    for doc in documents
    if doc["document"] == "FU_BF065_EN.pdf"
)

page_25 = next(
    page
    for page in prospectus["pages"]
    if page["page"] == 25
)

pd.DataFrame(page_25["text_blocks"])[
    ["text", "font_size", "bold", "bbox"]
].head(12)


## 4. Detect fund anchors in the full prospectus

The registry is already known at this point.

This step only asks: **where in the full prospectus does ownership change from one known fund to another?**


In [ ]:
anchors = fund_segmentation.find_fund_anchors(
    prospectus,
    registry,
)

anchor_df = pd.DataFrame(anchors)

display(anchor_df)
print("Detected anchors:", len(anchor_df))


## 5. Representative boundary evaluation

These are a small set of manually checked fund starts.

They are used only to evaluate segmentation. They are **not** provided to `find_fund_anchors`.


In [ ]:
GOLD_ANCHORS = [
    ("Enhanced Intelligence Global Allocation Responsible Investing", 25, "fund_section"),
    ("K&H egészség", 38, "fund_section"),
    ("K&H egészség 2", 50, "fund_section"),
    ("World Fix 100-1", 144, "fund_section"),
    ("World Fix 100-2", 156, "fund_section"),
    ("World Fix 100-3", 168, "fund_section"),
    ("World Fix 100-4", 180, "fund_section"),
    ("World Selection 100-1 Advanced", 204, "fund_section"),
    ("ČSOB Airbag Amerika 1", 219, "fund_section"),
    ("World Selection 100-1 Advanced", 515, "annex"),
]

predicted = {
    (item["fund"], item["page"], item["segment_type"])
    for item in anchors
}

checks = pd.DataFrame([
    {
        "fund": fund,
        "expected_page": page,
        "segment_type": segment_type,
        "found": (fund, page, segment_type) in predicted,
    }
    for fund, page, segment_type in GOLD_ANCHORS
])

display(checks)

recall = checks["found"].mean()
print(f"Representative border recall: {recall:.1%}")


## 6. Inspect possible extra anchors

Recall alone is not enough. This table helps us inspect whether segmentation created suspicious extra boundaries.

In particular, a repeated `fund_section` after an `annex` for the same fund may indicate a false boundary caused by a reference inside the annex.


In [ ]:
suspicious = []

for previous, current in zip(anchors, anchors[1:]):
    if (
        previous["segment_type"] == "annex"
        and current["segment_type"] == "fund_section"
        and previous["fund"] == current["fund"]
    ):
        suspicious.append({
            "fund": current["fund"],
            "annex_page": previous["page"],
            "later_fund_section_page": current["page"],
            "reason": "same fund re-detected as main section after annex",
        })

suspicious_df = pd.DataFrame(suspicious)

if suspicious_df.empty:
    print("No obvious annex-related false borders found.")
else:
    display(suspicious_df)
    print("Suspicious extra anchors:", len(suspicious_df))


## 7. Build and persist structural segments

No hand-written document-to-fund relationship map is injected here. General documents remain general (`fund=None`); fund-specific ownership is assigned only when supported by the discovery/segmentation structure.


In [ ]:
segments = fund_segmentation.segment_documents(
    documents,
    registry,
    classifications,
)

SEGMENT_PATH = SEGMENT_DIR / "segments.json"
SEGMENT_PATH.write_text(
    json.dumps(segments, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Segments:", len(segments))
print("Saved:", SEGMENT_PATH)


In [ ]:
segment_df = pd.DataFrame(segments)

prospectus_segments = segment_df[
    segment_df["document"] == "FU_BF065_EN.pdf"
].reset_index(drop=True)

display(prospectus_segments)
print("Prospectus segments:", len(prospectus_segments))


In [ ]:
[
    segment
    for segment in segments
    if segment["document"] == "11074.pdf"
]


## 8. Continuity and single-fund sanity checks


In [ ]:
prospectus_segment_records = [
    segment
    for segment in segments
    if segment["document"] == "FU_BF065_EN.pdf"
]

continuity_ok = all(
    (current["end_page"], current["end_block"])
    == (following["start_page"], following["start_block"])
    for current, following in zip(
        prospectus_segment_records,
        prospectus_segment_records[1:],
    )
)

print("Continuous / non-overlapping:", continuity_ok)

class_by_doc = {
    item["document"]: item
    for item in classifications
}

single_fund_checks = []

for document in documents:
    classification = class_by_doc.get(document["document"])

    if not classification or classification["document_type"] != "single_fund":
        continue

    doc_segments = [
        segment
        for segment in segments
        if segment["document"] == document["document"]
    ]

    single_fund_checks.append({
        "document": document["document"],
        "primary_fund": classification["primary_fund"],
        "segment_count": len(doc_segments),
        "correct": (
            len(doc_segments) == 1
            and doc_segments[0]["fund"] == classification["primary_fund"]
        ),
    })

display(pd.DataFrame(single_fund_checks))


## Conclusion

The last validated run detected the known prospectus fund boundaries with **100% representative boundary recall** and produced **44 continuous, non-overlapping prospectus segments**. Standalone single-fund documents each produced one segment owned by the expected fund, while the investor-rights document remained a general/no-fund segment.

The main limitation is structural dependence: this method assumes fund transitions are signaled by explicit headings or fund-list page references. If future prospectuses use weaker or inconsistent layouts, segmentation would need additional heading/entity evidence or a review fallback.
